# 11 – Recommendations

Esplorazione e data cleaning del dataset `recommendations.csv`.

| Colonna | Descrizione |
|---|---|
| `mal_id` | ID dell'anime su MAL |
| `recommendation_mal_id` | ID dell'anime raccomandato su MAL |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_rec = pd.read_csv('../datasets/recommendations.csv')
print(f'Shape: {df_rec.shape}')
df_rec.info()
df_rec.head()

**Osservazioni iniziali:**
- Il dataset contiene **105.249 righe** e **2 colonne**.
- Tutte e 2 le colonne sono complete: **nessun valore nullo**.
- I tipi di dati sono adeguati: `int64` per entrambi gli ID.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [ ]:
n_originale = len(df_rec)

mask_dup = df_rec.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_rec[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_rec.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_rec):,}')

Nessun duplicato esatto trovato. Tutte le 105.249 righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.

## 2. Analisi colonna per colonna

### 2.1 `mal_id`

Questa colonna è una **chiave esterna** che referenzia la chiave primaria `mal_id` di `details.csv`.

I valori duplicati sono **attesi**: lo stesso anime può comparire più volte come sorgente di più raccomandazioni diverse.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Usiamo quindi `check_fk` al posto di `analyze`, che effettua entrambi i controlli.

In [ ]:
df_details = pd.read_csv('../datasets_cleaned/details_clean.csv')

mask_orphan_mal = check_fk(df_rec['mal_id'], df_details['mal_id'], child_df=df_rec)

print(f'Null in mal_id               : {df_rec["mal_id"].isna().sum()}')
print(f'Duplicati in mal_id (attesi) : {df_rec["mal_id"].duplicated().sum():,}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime sorgente valido.
- **Integrità referenziale**: non ci sono righe orfane.

**Nessuna pulizia necessaria.**

### 2.2 `recommendation_mal_id`

Questa colonna è anch'essa una **chiave esterna** verso la stessa tabella padre `details.csv`, ma referenzia l'anime raccomandato.

I valori duplicati sono **attesi**: lo stesso anime può essere raccomandato a partire da diversi anime.

I controlli rilevanti sono:
- **Valori nulli**: una chiave esterna nulla indica una riga senza riferimento che va rimossa.
- **Integrità referenziale**: ogni ID presente qui deve esistere in `details_clean.csv`.

Riutilizziamo `df_details` già caricato nella sezione precedente.

In [ ]:
mask_orphan_rec = check_fk(df_rec['recommendation_mal_id'], df_details['mal_id'], child_df=df_rec)

print(f'Null in recommendation_mal_id: {df_rec["recommendation_mal_id"].isna().sum()}')
print(f'Duplicati in recommendation_mal_id (attesi): {df_rec["recommendation_mal_id"].duplicated().sum():,}')

**Osservazioni:**
- **Nessun valore nullo**: tutti i record hanno un ID anime raccomandato valido.
- **Integrità referenziale**: ci sono 127 righe orfane che vengono rimosse nella cella seguente.

In [ ]:
if mask_orphan_rec.any():
    n_orfane = mask_orphan_rec.sum()
    df_rec = df_rec[~mask_orphan_rec].reset_index(drop=True)
    print(f'Righe orfane rimosse : {n_orfane}')
    print(f'Righe rimanenti      : {len(df_rec):,}')
else:
    print('Nessuna riga orfana da rimuovere.')